# Colab: teacher Q&A then fp16 LoRA of Qwen3.5-2B

Run top to bottom on a free T4. Writes under Drive (`thesis-kd/`) so a disconnect loses at most the current row.

Resume: teacher and judge append JSONL. SFT uses `resume_from_checkpoint` when `sft-out/checkpoint-*` exists. The GGUF cell reloads `work/lora`.

This is response-level distillation: `qwen3.5:9b` writes grounded Q&A, then fp16 LoRA-SFT on the 2B. Not soft-target / logit KD. The method is LoRA, not QLoRA. The T4 has no bf16.

**Before you start**

1. Runtime → Change runtime type → T4 GPU.
2. On the laptop, rebuild `rag/lab-index` (default sources have no Ubuntu Server docs) and copy `chunks.jsonl` or `kd/seed_chunks.jsonl` to Drive `thesis-kd/`.
3. Write the held-out test questions first (`your-heldout.jsonl`). `make_seed_chunks.py --holdout` and `build_sft.py --holdout` need them so train rows do not leak the test set. Do not use `eval/test.jsonl`.
4. Default seed mix (no Server docs): tldr=460, command-manpages=295, man-pages=115, gnome-user-docs=230, ubuntu-docs=29. ubuntu-docs usually yields about 16 pages, so the written file is about 1,116 rows. Use the printed count, not the 1,129 mix total.


## Teacher path: Ollama, not transformers 4-bit

`kd/teacher_generate.py` and `kd/judge_rows.py` only call Ollama at `localhost:11434`. Ollama's `qwen3.5:9b` is already Q4 (~6.6–7.6 GB) and fits a T4 16 GB card. A transformers 4-bit teacher would be a second stack and would not reuse those scripts.

Set `USE_4B_TEACHER = True` if the 9B pull or load fails. That is a weaker teacher, not the same run. Stop Ollama before SFT so the 2B has the GPU.

The last cell prints the Modelfile and the laptop command `ollama create thesis-student -f Modelfile`.


In [ ]:
# Flags. Edit these, then run the rest in order.
REPO_URL = "https://github.com/Birdabo404/sui-SLM.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis-kd"
REPO_DIR = "/content/sui-SLM"
HELDOUT = ""  # empty: Drive your-heldout.jsonl, then repo eval/your-heldout.jsonl

USE_4B_TEACHER = False  # True -> qwen3.5:4b
USE_QLORA = False       # default is fp16 LoRA. T4 has no bf16. Unsloth advises against 4-bit on Qwen3.5.
DRY_RUN = False         # True: skip Ollama pull, teacher, and GPU SFT
TEACHER_LIMIT = 0       # 0 = all remaining seeds; 20 is a smoke pass
SAVE_EVERY = 10
EPOCHS = 1
MAX_SEQ = 2048
LORA_R = 16
BATCH = 1
GRAD_ACCUM = 8


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

work = Path(DRIVE_ROOT)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not Colab; using DRIVE_ROOT as a normal folder.")
    work.mkdir(parents=True, exist_ok=True)

work.mkdir(parents=True, exist_ok=True)
os.chdir(work)
print("Python", sys.version.split()[0])
print("work", work)


In [ ]:
repo = Path(REPO_DIR)
if not (repo / "kd" / "teacher_generate.py").is_file():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo)])
sys.path.insert(0, str(repo))
os.chdir(repo)
print("repo", repo)
print("head", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# Unsloth 2026.10.6 allows TRL after transformers 5. Colab does not auto-upgrade.
# Qwen3.5 tier is transformers 5.3.0. TRL 0.28.0 is the first release with
# transformers-5 SFT fixes that this Unsloth pin supports.
pkgs = [
    "sentence-transformers==3.3.1",
    "datasets",
    "peft",
    "accelerate",
    "bitsandbytes",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
pins = [
    "unsloth==2026.10.6",
    "unsloth_zoo==2026.10.6",
    "transformers==5.3.0",
    "trl==0.28.0",
]
if not DRY_RUN:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pins])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==5.3.0", "trl==0.28.0"])
print("pip ok")


In [ ]:
from kd.colab_helpers import (
    ASSISTANT_PREFIX,
    CHATML_INSTRUCTION,
    CHATML_RESPONSE,
    OLLAMA_NAME,
    STUDENT_HF,
    count_jsonl,
    latest_sft_checkpoint,
    ollama_create_cmd,
    ollama_tags_ok,
    render_chatml,
    require_seeds,
    resolve_chunks,
    resolve_heldout,
    resolve_seeds,
    select_q4km_gguf,
    teacher_ollama_name,
    write_modelfile,
)

heldout = resolve_heldout(
    [HELDOUT, work / "your-heldout.jsonl", repo / "eval" / "your-heldout.jsonl"],
    work,
)
print("heldout", heldout)


In [ ]:
seeds = resolve_seeds(work, repo)
if seeds is None:
    chunks = resolve_chunks(work, repo)
    if chunks is None:
        require_seeds(work, repo)
    seeds = work / "seed_chunks.jsonl"
    subprocess.check_call([
        sys.executable, str(repo / "kd" / "make_seed_chunks.py"),
        "--chunks", str(chunks),
        "--out", str(seeds),
        "--holdout", str(heldout),
    ])
elif seeds.resolve() != (work / "seed_chunks.jsonl").resolve():
    shutil.copy2(seeds, work / "seed_chunks.jsonl")
    seeds = work / "seed_chunks.jsonl"
n_seeds = count_jsonl(seeds)
print("seeds", seeds, "rows", n_seeds)
print("Use this printed count in Ch3 (about 1,116). Do not report the 1,129 mix target.")


In [ ]:
teacher = teacher_ollama_name(USE_4B_TEACHER)
print("teacher", teacher, "(4B fallback)" if USE_4B_TEACHER else "(9B)")
if DRY_RUN:
    print("DRY_RUN: skip Ollama install/pull")
else:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "zstd", "pciutils"])
    if shutil.which("ollama") is None:
        subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])
    if ollama_tags_ok():
        print("Ollama already responding at /api/tags; skip serve")
    else:
        log = work / "ollama.log"
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=open(log, "ab"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        for _ in range(30):
            if ollama_tags_ok():
                break
            time.sleep(2)
        else:
            raise SystemExit("Ollama did not start. See " + str(log))
    subprocess.check_call(["ollama", "pull", teacher])


In [ ]:
raw = work / "teacher_raw.jsonl"
cmd = [
    sys.executable, str(repo / "kd" / "teacher_generate.py"),
    "--seeds", str(seeds),
    "--out", str(raw),
    "--model", teacher,
]
if TEACHER_LIMIT:
    cmd.extend(["--limit", str(TEACHER_LIMIT)])
if DRY_RUN:
    print("DRY_RUN: skip teacher_generate.", "already", count_jsonl(raw))
else:
    print("resume teacher; already", count_jsonl(raw), "flush every row, status every", SAVE_EVERY)
    subprocess.check_call(cmd)
print("teacher_raw", raw, "rows", count_jsonl(raw))


In [ ]:
judged = work / "teacher_judged.jsonl"
if DRY_RUN:
    print("DRY_RUN: skip judge_rows.", "already", count_jsonl(judged))
else:
    subprocess.check_call([
        sys.executable, str(repo / "kd" / "judge_rows.py"),
        "--raw", str(raw),
        "--seeds", str(seeds),
        "--out", str(judged),
        "--model", teacher,
    ])
print("teacher_judged", judged, "rows", count_jsonl(judged))


In [ ]:
raw_n = count_jsonl(raw)
judged_n = count_jsonl(judged) if judged.is_file() else 0
if judged_n and judged_n < raw_n:
    raise SystemExit(
        f"teacher_judged.jsonl has {judged_n} rows but teacher_raw.jsonl has {raw_n}. "
        "Re-run the judge cell so every raw row is judged."
    )
sft_in = judged if judged_n else raw
train_path = work / "sft_train.jsonl"
val_path = work / "sft_val.jsonl"
if not sft_in.is_file() or count_jsonl(sft_in) == 0:
    raise SystemExit("No teacher rows to build SFT from: " + str(sft_in))
subprocess.check_call([
    sys.executable, str(repo / "kd" / "build_sft.py"),
    "--raw", str(sft_in),
    "--seeds", str(seeds),
    "--holdout", str(heldout),
    "--out-train", str(train_path),
    "--out-val", str(val_path),
])
print("train", count_jsonl(train_path), "val", count_jsonl(val_path))


In [ ]:
# Free the GPU before SFT. Teacher and student should not share the T4.
if not DRY_RUN and shutil.which("ollama"):
    subprocess.run(["pkill", "-f", "ollama"], check=False)
    print("stopped Ollama")


In [ ]:
# Cheap smoke: build SFTConfig and check the ChatML prefix before the long run.
from trl import SFTConfig

sft_args = SFTConfig(
    output_dir=str(work / "sft-out"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=50,
    max_length=MAX_SEQ,
    dataset_text_field="text",
    optim="adamw_8bit",
    seed=3407,
    report_to=[],
)
if getattr(sft_args, "assistant_only_loss", False):
    raise SystemExit("assistant_only_loss must stay off; Qwen3.5 has no {% generation %} blocks.")
print("SFTConfig ok", "max_length", sft_args.max_length, "optim", sft_args.optim)
print("ChatML instruction", repr(CHATML_INSTRUCTION), "response", repr(CHATML_RESPONSE))
if train_path.is_file() and count_jsonl(train_path):
    row = json.loads(next(l for l in open(train_path, encoding="utf-8") if l.strip()))
    rendered = row.get("text") or render_chatml(row["messages"])
    if ASSISTANT_PREFIX not in rendered:
        raise SystemExit("SFT text is missing the empty think prefix " + repr(ASSISTANT_PREFIX))
    print("SFT text prefix matches Modelfile")


In [ ]:
# fp16 LoRA SFT of unsloth/Qwen3.5-2B. FastLanguageModel is the text-only loader.
# Loss is only on the assistant answer (train_on_responses_only).
if DRY_RUN:
    print("DRY_RUN: skip GPU SFT. train rows", count_jsonl(train_path))
    if train_path.is_file() and count_jsonl(train_path):
        row = json.loads(next(l for l in open(train_path, encoding="utf-8") if l.strip()))
        print("sample roles", [m["role"] for m in row["messages"]])
else:
    if USE_QLORA:
        print("USE_QLORA=True. Unsloth reports large 4-bit error on Qwen3.5; default is fp16 LoRA.")
    try:
        from unsloth import FastLanguageModel
        use_unsloth = True
    except ImportError:
        use_unsloth = False
        print("unsloth missing; peft + bitsandbytes fallback")

    from datasets import load_dataset
    from trl import SFTTrainer

    data = load_dataset("json", data_files={"train": str(train_path), "val": str(val_path)})
    out_dir = work / "sft-out"
    if use_unsloth:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=STUDENT_HF,
            max_seq_length=MAX_SEQ,
            load_in_4bit=USE_QLORA,
            load_in_16bit=not USE_QLORA,
            full_finetuning=False,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=LORA_R,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
            max_seq_length=MAX_SEQ,
        )
    else:
        import torch
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        tokenizer = AutoTokenizer.from_pretrained(STUDENT_HF, trust_remote_code=True)
        if USE_QLORA:
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            model = AutoModelForCausalLM.from_pretrained(
                STUDENT_HF, quantization_config=bnb, device_map="auto", trust_remote_code=True,
            )
            model = prepare_model_for_kbit_training(model)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                STUDENT_HF, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
            )
        model = get_peft_model(model, LoraConfig(
            r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none", task_type="CAUSAL_LM",
        ))

    processing = getattr(tokenizer, "tokenizer", None) or tokenizer
    trainer = SFTTrainer(
        model=model,
        processing_class=processing,
        train_dataset=data["train"],
        eval_dataset=data["val"] if count_jsonl(val_path) else None,
        args=sft_args,
    )
    if use_unsloth:
        from unsloth.chat_templates import train_on_responses_only
        trainer = train_on_responses_only(
            trainer,
            instruction_part=CHATML_INSTRUCTION,
            response_part=CHATML_RESPONSE,
        )
    resume = latest_sft_checkpoint(out_dir)
    if resume:
        print("resume SFT from", resume)
    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)
    trainer.save_model(str(work / "lora"))
    processing.save_pretrained(str(work / "lora"))
    loss_path = work / "sft_loss.json"
    hist = result.metrics
    loss_path.write_text(json.dumps(hist, indent=2), encoding="utf-8")
    print("loss metrics", hist)
    print("wrote", loss_path)


In [ ]:
gguf_dir = work / "gguf"
gguf_dir.mkdir(parents=True, exist_ok=True)
lora_dir = work / "lora"
if DRY_RUN:
    print("DRY_RUN: skip GGUF")
else:
    if not lora_dir.is_dir() or not any(lora_dir.iterdir()):
        raise SystemExit("No LoRA adapter at " + str(lora_dir) + ". Run the SFT cell first.")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(lora_dir),
        max_seq_length=MAX_SEQ,
        load_in_4bit=False,
        load_in_16bit=True,
        full_finetuning=False,
    )
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q4_k_m")
    print("gguf dir", list(gguf_dir.glob("*"))[:20])


In [ ]:
if DRY_RUN:
    print("DRY_RUN: no GGUF yet. The Modelfile cell needs a real *Q4_K_M*.gguf; it will not invent a name.")
else:
    gguf_path = select_q4km_gguf(gguf_dir)
    mf = write_modelfile(work / "Modelfile", gguf_path.name)
    print(mf.read_text())
    print()
    print("On the laptop, copy the GGUF and Modelfile into one folder, then:")
    print("  " + ollama_create_cmd(OLLAMA_NAME))
    print("Eval:")
    print(
        "  python eval/run_eval.py --split test --no-sweep "
        "--questions eval/your-heldout.jsonl "
        "--out logs/eval-test.csv --distilled " + OLLAMA_NAME
    )
    print("gguf", gguf_path)
print("USE_4B_TEACHER", USE_4B_TEACHER, "USE_QLORA", USE_QLORA, "DRY_RUN", DRY_RUN)
